# Practice 2: Pre-trained Neural Networks (Transfer Learning)

Welcome to the Research Notebook for **Practice 2**. This notebook serves as an interactive report detailing our approach to Transfer Learning on the CIFAR-10 dataset using pre-trained neural networks (like ResNet18 and VGG16).

## 1. Project Introduction

In this project, we implement a production-ready Deep Learning pipeline focusing on:
- **Clean Architecture**: Separation of concerns (Data, Model, Train, Evaluate, Visualize).
- **Reproducibility**: Seed fixing and deterministic execution.
- **Experiment Tracking**: Managing different transfer learning strategies (Head-only vs Partial Fine-Tuning).
- **Visualization**: Comprehensive reporting on data distribution, learning curves, and model confidence.

## 2. Project Structure
The core logic is modularized in the `processing_own_phase` package. We will not run training loops here; instead, we import the modules to visualize and analyze the setup.

In [1]:
import os
import sys
import pandas as pd
from pathlib import Path
from IPython.display import Image, display, Markdown

# Add parent directory to sys.path so we can import from root
sys.path.append(os.path.abspath('..'))

# Set up paths
from configs import CONFIG, CLASS_NAMES, EXPERIMENTS, REPORTS_DIR, OUTPUT_DIR

print("Modules successfully imported from source!")

Modules successfully imported from source!


## 3. Dataset Overview
We use the **CIFAR-10** dataset. Let's look at the class distribution.

In [2]:
from processing_own_phase.data import load_datasets, get_class_distribution
from processing_own_phase.visualize import plot_class_distribution, plot_class_examples

# Load datasets to inspect
train_subset, val_subset, test_dataset = load_datasets(CONFIG["data_dir"], 1.0 - CONFIG["train_split_ratio"], CONFIG["seed"])

print(f"Number of classes: {len(CLASS_NAMES)}")
print(f"Class Names: {CLASS_NAMES}")
print(f"Train size: {len(train_subset)} | Val size: {len(val_subset)} | Test size: {len(test_dataset)}")

Number of classes: 10
Class Names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train size: 45001 | Val size: 4999 | Test size: 10000


## 4. Data Pipeline & Visualization
Here we show the actual distribution of the dataset splits and a few sample images per class.

In [3]:
distributions = {
    "Train": get_class_distribution(train_subset),
    "Val":   get_class_distribution(val_subset),
    "Test":  get_class_distribution(test_dataset),
}

# Note: Calling these functions generates the plot and we display it directly
plot_class_distribution(distributions, save_path=None, show=True)
plot_class_examples(train_subset, CLASS_NAMES, num_examples=5, save_path=None, show=True)

d:\Hoc_tap\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_2\processing_own_phase\visualize.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Model Architecture
We use pre-trained models. Let's inspect the `resnet18` model in `head_only` mode.

In [4]:
from processing_own_phase.model import build_model
import torch

model = build_model("resnet18", training_mode="head_only")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: ResNet-18 (Pre-trained)")
print(f"Strategy: Freeze backbone (head_only)")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {total_params - trainable_params:,}")

# Display the classification head
print("\nClassification Head:")
print(model.network.fc)

Model: ResNet-18 (Pre-trained)
Strategy: Freeze backbone (head_only)
Total Parameters: 11,181,642
Trainable Parameters: 5,130
Frozen Parameters: 11,176,512

Classification Head:
Linear(in_features=512, out_features=10, bias=True)


## 6. Training Configuration
The baseline training configuration is loaded directly from `CONFIG`.

In [5]:
config_df = pd.DataFrame(list(CONFIG.items()), columns=["Parameter", "Value"])
display(config_df)

,Parameter,Value
0,seed,42
1,batch_size,64
2,num_workers,0
3,train_split_ratio,0.9
4,image_size,224
5,epochs,10
6,optimizer,Adam
7,learning_rate,0.001
8,weight_decay,0.0001
9,scheduler,ReduceLROnPlateau


## 7. Experiment Configuration
We have set up several experiments varying the architecture and the transfer learning strategy.

In [6]:
exp_df = pd.DataFrame.from_dict(EXPERIMENTS, orient='index')
display(exp_df[["model_name", "training_mode", "optimizer", "learning_rate", "epochs", "description"]])

,model_name,training_mode,optimizer,learning_rate,epochs,description
E1_resnet18_head,resnet18,head_only,Adam,0.0010,10,"ResNet18 Transfer Learning: Freeze backbone, t..."
E2_resnet18_partial,resnet18,partial_finetune,Adam,0.0005,10,ResNet18 Partial Fine Tuning: Unfreeze last bl...
E3_vgg16_head,vgg16,head_only,Adam,0.0010,5,VGG16 Transfer Learning.
E4_mobilenet_partial,mobilenet_v3_small,partial_finetune,AdamW,0.0005,5,MobileNetV3 Small Partial Fine Tuning.


## 8. Evaluation Results
After running `python -m processing_own_phase.main`, the pipeline saves an `experiment_results.csv` and a `summary.json`. Let's view the comparison table.

In [7]:
comparison_csv = OUTPUT_DIR / "comparison_table.csv"
if comparison_csv.exists():
    results_df = pd.read_csv(comparison_csv)
    display(results_df)
else:
    print("Run the pipeline first to generate the comparison_table.csv.")

Run the pipeline first to generate the comparison_table.csv.


## 9. Visualization
The visualization module generates multiple plots assessing model performance. We display the generated PNG files here.

In [8]:
def show_report_image(filename, title):
    path = REPORTS_DIR / filename
    if path.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(path)))
    else:
        print(f"Missing {filename}. Please run the pipeline.")

show_report_image("training_curves.png", "Training and Validation Curves")
show_report_image("learning_rate.png", "Learning Rate Schedule")
show_report_image("confusion_matrix_normalized.png", "Normalized Confusion Matrix")
show_report_image("metrics_bar.png", "Metrics per Class")
show_report_image("experiment_comparison.png", "Experiment Comparison")
show_report_image("prediction_gallery_correct.png", "Top Confident Correct Predictions")
show_report_image("prediction_gallery_incorrect.png", "Top Confident Incorrect Predictions")

Missing training_curves.png. Please run the pipeline.
Missing learning_rate.png. Please run the pipeline.
Missing confusion_matrix_normalized.png. Please run the pipeline.
Missing metrics_bar.png. Please run the pipeline.
Missing experiment_comparison.png. Please run the pipeline.
Missing prediction_gallery_correct.png. Please run the pipeline.
Missing prediction_gallery_incorrect.png. Please run the pipeline.


## 10. TensorBoard

To view real-time logs, histograms, and the computation graph, run the following command in your terminal:

```bash
tensorboard --logdir=runs
```

**What to look for:**
- **Scalars**: Loss and Accuracy tracking per epoch.
- **Histograms**: Weight and gradient distributions to check for exploding/vanishing gradients.
- **Graphs**: The neural network topology.

## 11. Conclusion
Let's parse the final `summary.json` to extract the key takeaways from the execution.

In [9]:
import json

summary_json = OUTPUT_DIR / "summary.json"
if summary_json.exists():
    with open(summary_json, "r") as f:
        summary = json.load(f)
    
    display(Markdown(f"**Best Experiment**: `{summary['best_experiment']}`"))
    display(Markdown(f"**Best Validation Accuracy**: `{summary['best_val_acc']:.2f}%`"))
    display(Markdown(f"**Test Accuracy**: `{summary['test_accuracy']*100:.2f}%`"))
    display(Markdown(f"**Macro F1-Score**: `{summary['macro_f1']:.4f}`"))
    display(Markdown(f"**Inference FPS**: `{summary['inference_fps']:.1f}`"))
else:
    print("Summary not found. Please run the pipeline.")

Summary not found. Please run the pipeline.
